# Сравнение на основе сохраненных метрик (JSON)


In [9]:
import pandas as pd
import plotly.express as px

In [10]:
import json
from pathlib import Path

base_path = Path('/Users/mac/Documents/Analise Dushanbe school, ML, Article/Models/Compare models')
logreg_path = base_path / 'logreg_metrics.json'
cat_path = base_path / 'catboost_metrics.json'

logreg_metrics = json.loads(logreg_path.read_text())
cat_metrics = json.loads(cat_path.read_text())

metrics_df = pd.DataFrame([logreg_metrics, cat_metrics])
metrics_df

,model,AUC,Accuracy,Precision,Recall,F1,CV_AUC_mean,CV_AUC_std,excluded_leakage_cols,threshold,threshold_strategy,min_precision_constraint
0,LogReg,0.828924,0.734426,0.826347,0.726316,0.773109,0.780279,0.017239,[Средний_балл],NaN,NaN,NaN
1,CatBoost,0.845767,0.806557,0.801843,0.915789,0.855037,NaN,NaN,NaN,0.35,maximize_recall_with_min_precision,0.8


In [11]:
# Визуализация сохраненных метрик (устойчиво к разным ключам JSON)
# Берем только общие и числовые метрики для честного сравнения.
metric_order = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']

available_metrics = [
    m for m in metric_order
    if m in metrics_df.columns and pd.api.types.is_numeric_dtype(metrics_df[m])
]

if not available_metrics:
    raise ValueError('Не найдены общие числовые метрики для сравнения.')

metrics_plot_df = metrics_df[['model'] + available_metrics].copy()
metrics_long = metrics_plot_df.melt(id_vars='model', var_name='metric', value_name='value')

fig = px.bar(
    metrics_long,
    x='metric', y='value', color='model', barmode='group', text='value',
    category_orders={'metric': available_metrics},
    title='Сравнение ключевых метрик моделей'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0, 1]))
fig.show()

# Доп. таблица: поля, которые не участвуют в прямом сравнении
extra_cols = [c for c in metrics_df.columns if c not in ['model'] + available_metrics]
if extra_cols:
    print('Служебные/дополнительные поля (не для bar-сравнения):')
    display(metrics_df[['model'] + extra_cols])


Служебные/дополнительные поля (не для bar-сравнения):


,model,CV_AUC_mean,CV_AUC_std,excluded_leakage_cols,threshold,threshold_strategy,min_precision_constraint
0,LogReg,0.780279,0.017239,[Средний_балл],NaN,NaN,NaN
1,CatBoost,NaN,NaN,NaN,0.35,maximize_recall_with_min_precision,0.8


In [12]:
# Числовая таблица по среднему скору только по ключевым метрикам
metric_order = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
available_metrics = [
    m for m in metric_order
    if m in metrics_df.columns and pd.api.types.is_numeric_dtype(metrics_df[m])
]

metrics_df['mean_score'] = metrics_df[available_metrics].mean(axis=1)
metrics_df.sort_values('mean_score', ascending=False)


,model,AUC,Accuracy,Precision,Recall,F1,CV_AUC_mean,CV_AUC_std,excluded_leakage_cols,threshold,threshold_strategy,min_precision_constraint,mean_score
1,CatBoost,0.845767,0.806557,0.801843,0.915789,0.855037,NaN,NaN,NaN,0.35,maximize_recall_with_min_precision,0.8,0.844999
0,LogReg,0.828924,0.734426,0.826347,0.726316,0.773109,0.780279,0.017239,[Средний_балл],NaN,NaN,NaN,0.777825


## Вывод

По результатам сравнения `LogReg` показывает более сильный общий баланс качества: выше `AUC`, `Accuracy`, `Recall`, `F1` и итоговый `mean_score`. `CatBoost` превосходит только по `Precision`, поэтому если приоритет — минимизация ложноположительных срабатываний, его можно рассматривать как альтернативу. В рамках текущих данных базовой моделью стоит оставить `LogReg`.